# 03 — Merge Features
Combines all processed data sources into one flat training CSV.
Output: `data/processed/training_features.csv`

Required inputs:
- `data/processed/quickstats_yield.csv` (from 01)
- `data/processed/weather_features.csv` (from 02)
- `data/raw/ndvi_by_state_date.csv` (from 03_satellite — optional, add when AWS available)

In [ ]:
import pandas as pd

# Load processed sources
yield_df = pd.read_csv("../data/processed/quickstats_yield.csv")
weather_df = pd.read_csv("../data/processed/weather_features.csv")

print(f"Yield rows: {len(yield_df)}")
print(f"Weather rows: {len(weather_df)}")

In [ ]:
# Merge on year + state
df = yield_df.merge(weather_df, on=['year', 'state'], how='left')
print(f"Merged shape: {df.shape}")
print(df.head())

In [ ]:
# Merge NDVI — required for pipeline
import os

ndvi_path = "../data/raw/ndvi_combined.csv"
if os.path.exists(ndvi_path):
    ndvi_df = pd.read_csv(ndvi_path)
    df = df.merge(ndvi_df, on=['year', 'state'], how='left')
    print(f"NDVI merged. Shape: {df.shape}")
else:
    raise FileNotFoundError(
        "ndvi_combined.csv not found. Run 03_satellite.ipynb (SageMaker) and "
        "03b_modis_backfill.ipynb, then re-run this notebook."
    )

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Drop rows with missing yield (our Y variable must be present)
df = df.dropna(subset=['yield_bu_acre'])

# Save master training CSV
df.to_csv("../data/processed/training_features.csv", index=False)
print(f"\nSaved {len(df)} rows to training_features.csv")
print(df.columns.tolist())